# ⛰ Core Lab 9 — Keep the GPU Busy
**A Billion Small Steps · Hardware for AI · Step 4 · Book sections 4.4, 4.C · Foothold lessons 4.4, 4.C**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** an "impossible" time without synchronisation; the cost of `.item()`; a profiled training loop made several times faster.

| Part | What you do | Book |
|---|---|---|
| A | The impossible timing, and the cost of `.item()` | 4.4 |
| B | Profile and fix a slow training loop | 4.C |

⏱️ About 2 hours. 📘 Textbook: Hwu, Kirk & El Hajj Ch 2 · Hamacher Ch 3. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

## Part A · Asynchronous Launches  *(Section 4.4)*

First, see the "impossible" timing for yourself.

In [ ]:
# two big matrices on the GPU, and one warm-up multiply
a = torch.rand(4096, 4096, device=device); b = torch.rand(4096, 4096, device=device)
a @ b; sync()   # a warm-up multiply

# WRONG: stop the clock as soon as Python returns, when the GPU has only been GIVEN the order
t0 = time.perf_counter(); c = a @ b; t_wrong = time.perf_counter() - t0
# let that multiply finish before the next test
sync()
# RIGHT: wait for the GPU to finish (sync) before stopping the clock
t0 = time.perf_counter(); c = a @ b; sync(); t_right = time.perf_counter() - t0

# the work in one 4096 × 4096 multiply
flops = 2 * 4096**3
print(f"WITHOUT sync: {t_wrong*1e3:8.3f} ms  → 'would be' {flops / t_wrong / 1e12:8.1f} TFLOPS  (impossible)")   # the misleading number
print(f"WITH sync   : {t_right*1e3:8.3f} ms  → {flops / t_right / 1e12:8.1f} TFLOPS  (real)")   # the honest number

Now the cost of a **sync point every step**: train a small network for 300 steps, calling `loss.item()` every step vs only at the end.

In [ ]:
# a small network on the GPU, and one fixed batch of made-up data: 2,048 inputs of 1,024 numbers, 2,048 labels
net = torch.nn.Sequential(torch.nn.Linear(1024, 1024), torch.nn.ReLU(), torch.nn.Linear(1024, 10)).to(device)
opt = torch.optim.SGD(net.parameters(), lr=0.01)   # plain gradient descent
X = torch.randn(2048, 1024, device=device); Y = torch.randint(0, 10, (2048,), device=device)   # one fixed batch of made-up data, on the GPU
lossf = torch.nn.CrossEntropyLoss()   # the loss for "which class?" problems

def prepare_next_batch():   # a stand-in for the CPU's own work each step (reading and preparing the next batch)
    # about 2 ms of plain-Python arithmetic, done by the CPU alone
    return sum(i * i for i in range(20_000))

def train(steps, item_every_step):   # train for some steps, with or without .item() every step
    # a running total that lives on the GPU, and a Python list for the every-step version
    total = torch.zeros((), device=device); log = []
    for _ in range(steps):   # one training step each time round
        # forward pass + loss, then slopes + update: all queued for the GPU
        loss = lossf(net(X), Y)
        opt.zero_grad(); loss.backward(); opt.step()   # clear old slopes, backward pass, update
        if item_every_step:   # the version with a sync point every step
            # .item() copies the number to the CPU, so the CPU must wait for the GPU: a sync every step
            log.append(loss.item())
        else:   # the version without
            # .detach() keeps the value without its history; it stays on the GPU, no waiting
            total += loss.detach()
        # the CPU prepares the next batch: while the GPU is still training, unless the CPU just waited for it
        prepare_next_batch()
    # one sync at the end
    return total.item()

item_ms = {}   # True/False → time in ms
for flag in [True, False]:   # first with .item() every step, then without
    # time 300 steps three times and keep the best (Colab's shared CPU makes single timings jumpy); × 1e3 → ms
    item_ms[flag] = min(timeit(lambda: train(300, flag), repeats=1) for _ in range(3)) * 1e3
    print(f"item() every step = {flag!s:5s}: {item_ms[flag]:7.1f} ms for 300 steps")   # show both

### ✏️ YOUR TURN (A)
1. How many times "faster than peak" was the unsynchronised timing? What was really being measured?
2. How much slower was calling `.item()` every step? (The gap grows when the CPU has other work to overlap, as in Part B.)

*Your answers:*

## Part B · Profile and Fix a Slow Training Loop  *(Section 4.C)*

The final-year team's loop vs the fixed loop, on MNIST (the handwritten digits from Core Lab 3), one epoch each.

In [ ]:
from torch import nn   # nn: PyTorch's layers and losses
from torchvision import datasets, transforms   # ready-made datasets, and ways to convert images
from torch.utils.data import DataLoader   # DataLoader serves a dataset in batches
# scikit-learn's accuracy function (it works on CPU NumPy arrays)
from sklearn.metrics import accuracy_score

# MNIST handwritten digits, turned into tensors with pixel values 0..1
train_data = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())

def make_model():   # build a fresh copy of the network
    # the same starting weights for both loops: a fair race
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10)).to(device)   # 784 → 512 → 512 → 10, moved to the GPU

def slow_epoch(max_steps=None):   # one epoch of the team's loop (max_steps lets the profiler stop early)
    # the team's original loop: batches of 16, no worker processes
    model = make_model(); opt = torch.optim.SGD(model.parameters(), lr=0.05); lossf = nn.CrossEntropyLoss()
    loader = DataLoader(train_data, batch_size=16, shuffle=True)   # batches of 16, shuffled, no helper processes
    history = []   # the loss and accuracy of every step
    for step, (images, labels) in enumerate(loader):   # one batch at a time; enumerate also counts the steps
        images, labels = images.to(device), labels.to(device)   # copy the batch to the GPU
        scores = model(images); loss = lossf(scores, labels)   # forward pass and loss
        opt.zero_grad(); loss.backward(); opt.step()   # clear old slopes, backward pass, update
        # accuracy on the CPU: copies out and a sync, every step
        acc = accuracy_score(labels.cpu().numpy(), scores.argmax(1).cpu().numpy())
        # and another sync, every step
        history.append((loss.item(), acc))
        if max_steps and step + 1 >= max_steps: break   # stop early if asked
    return float(np.mean([h[0] for h in history])), float(np.mean([h[1] for h in history]))   # the epoch's average loss and accuracy

def fast_epoch(max_steps=None):   # one epoch of the fixed loop
    # the fixed loop: same model and data, better habits; a bigger batch can take a bigger learning rate
    model = make_model(); opt = torch.optim.SGD(model.parameters(), lr=0.3); lossf = nn.CrossEntropyLoss()
    # batches of 256; 2 worker processes prepare batches in parallel; pinned memory for faster copies
    loader = DataLoader(train_data, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
    # totals kept ON the GPU
    running = torch.zeros((), device=device); correct = torch.zeros((), device=device)
    for step, (images, labels) in enumerate(loader):   # one batch at a time; enumerate also counts the steps
        # non_blocking=True: start the copy and carry on without waiting for it
        images = images.to(device, non_blocking=True); labels = labels.to(device, non_blocking=True)
        scores = model(images); loss = lossf(scores, labels)   # forward pass and loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()   # clear old slopes (the cheapest way), backward pass, update
        # no .item(): nothing makes the CPU wait
        running += loss.detach(); correct += (scores.argmax(1) == labels).sum()
        if max_steps and step + 1 >= max_steps: break   # stop early if asked
    seen = min((step + 1) * 256, len(train_data))   # how many images we went through
    # ONE sync, at the very end
    return running.item() / (step + 1), correct.item() / seen

epoch_s = {}   # loop name → seconds per epoch
for name, fn in [("SLOW (team's loop)", slow_epoch), ("FAST (fixed loop)", fast_epoch)]:   # race the two loops
    # time one full epoch of each
    sync(); t0 = time.perf_counter(); out = fn(); sync()
    epoch_s[name] = time.perf_counter() - t0   # keep the time
    print(f"{name:20s}: {epoch_s[name]:6.1f} s per epoch   (epoch-average loss {out[0]:.3f}, accuracy {out[1]:.1%})")   # show the time, the loss and the accuracy

**Fair comparison note:** after one epoch the fast loop's accuracy may be a little lower. Batch 256 makes only **235 weight updates per epoch** compared with **3,750** for batch 16 (Section 2.2: epochs, batches, iterations). Train the fast loop for 2–3 epochs and it catches up, still in less total time. Faster per epoch is only useful if the model still learns well, so always check both.

**The professional tool: the PyTorch profiler.** Profile 30 steps of each loop and compare the top entries. Look for `cudaMemcpyAsync` (copies) and `cudaStreamSynchronize` (the CPU waiting).

In [ ]:
# PyTorch's built-in profiler: record what the CPU and the GPU do
from torch.profiler import profile, ProfilerActivity
acts = [ProfilerActivity.CPU, ProfilerActivity.CUDA]   # record what the CPU and the GPU do
for name, fn in [("SLOW", slow_epoch), ("FAST", fast_epoch)]:   # profile both loops
    # everything inside this block is recorded: just 30 steps of the loop
    with profile(activities=acts) as prof:
        fn(max_steps=30)   # just 30 steps of the loop
    print(f"===== {name} loop, 30 steps =====")   # a heading for this loop
    # every operation the profiler saw, by name
    rows = {e.key: e for e in prof.key_averages()}
    # three rows to compare: the CPU waiting for the GPU, copies, and launching kernels
    for key, meaning in [("cudaStreamSynchronize", "CPU waits for the GPU"), ("cudaMemcpyAsync", "a copy"), ("cudaLaunchKernel", "start a kernel")]:
        # this row's numbers (zero if the loop never did it)
        ms, n = (rows[key].self_cpu_time_total / 1e3, rows[key].count) if key in rows else (0.0, 0)
        # its name, what it means, CPU time in ms, and how often it ran
        print(f"  {key:22s} ({meaning:21s}) {ms:7.1f} ms   called {n:4d} times")

### ✏️ YOUR TURN (B) — record your results
| | Slow loop | Fast loop |
|---|---|---|
| Seconds per epoch | | |
| Batch size | 16 | 256 |
| Sync points per step | | |

1. What speed-up did the fixes give?
2. In the profiler lists, name one entry that's big in the SLOW loop but small in the FAST loop, and explain it using a Step 4 idea.
3. Run `!nvidia-smi` in a new cell **while** the slow loop runs (start it, then quickly run the other cell). What GPU-Util % do you see?

*Your answers:*

---
## 🎓 Wrap-up
- GPU launches are asynchronous: time GPU work only after `torch.cuda.synchronize()`.
- Every `.item()`, `print` of a GPU value or `.cpu()` is a sync point that makes the CPU wait.
- Profile first, then fix the biggest cost: here, batch size, data on the GPU and fewer sync points.
- **Next — Core Lab 10: Make It Fast.** Tiling, low precision and your own roofline.